In [2]:
import numpy as np
import pandas as pd

In [26]:
import pandas as pd

# Load dataset
df = pd.read_csv("eeg_feature_dataset.csv")

# Drop rows where Label == 1
df = df[df["Label"] != 1]

# Rename Label 2 → 1
df["Label"] = df["Label"].replace({2: 1})

# Reset index
df = df.reset_index(drop=True)

print(df["Label"].value_counts())


Label
0.0    1600
1.0     800
Name: count, dtype: int64


In [27]:
X = df.drop('Label',axis=1)
y = df['Label']

In [28]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

In [29]:
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_scaled)

In [30]:
num_components = X_reduced.shape[1]
pca_columns = [f'PC{i+1}' for i in range(num_components)]
X_reduced_df = pd.DataFrame(X_reduced, columns=pca_columns)

In [31]:
df_pca = X_reduced_df
df_pca = df_pca.join(y)

In [32]:
df_pca

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,Label
0,-1.495073,-0.620827,0.674047,0.371953,0.731604,0.224873,-0.218734,0.740989,-0.033283,-0.296972,0.0
1,-2.081460,-0.100519,0.944545,0.953272,0.034408,0.413089,-0.222423,0.636461,0.049128,-0.687259,0.0
2,-1.377114,-0.884713,1.403403,0.494852,-0.192491,-0.073384,-0.288193,0.502305,-0.494391,-0.393831,0.0
3,-0.559787,-1.872423,1.381706,0.112241,0.197035,0.320140,-0.018371,0.362322,-0.662958,-0.315035,0.0
4,1.144167,-2.963009,1.109124,-0.214661,0.221964,-0.247762,0.348621,0.165121,-0.435596,0.227098,0.0
...,...,...,...,...,...,...,...,...,...,...,...
2395,1.662384,-2.232698,0.501272,-0.735863,-0.187630,-0.092863,0.168381,0.468263,0.091269,0.169536,1.0
2396,1.663277,-1.459597,-0.276317,0.110595,0.374228,-0.307580,0.345776,0.820580,0.685578,-0.071950,1.0
2397,1.247597,-1.483581,0.251585,-0.069624,-0.247199,-0.065858,0.299693,0.726175,0.533398,0.049787,1.0
2398,1.699740,-1.847167,0.195775,-0.441797,0.041011,-0.268193,0.226873,0.547839,0.531669,0.231145,1.0


In [33]:
import lightgbm as lgb
import pandas as pd
from sklearn.model_selection import train_test_split

X = df_pca.drop('Label',axis=1)
y=df_pca['Label']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = lgb.LGBMClassifier(random_state=42)
model.fit(X_train, y_train)

importances = model.feature_importances_
feature_names = X.columns

feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

top_50_features = feat_imp_df['feature'].head(50).tolist()
X_top50light = X[top_50_features]


[LightGBM] [Info] Number of positive: 643, number of negative: 1277
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000139 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 1920, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.334896 -> initscore=-0.686124
[LightGBM] [Info] Start training from score -0.686124


In [34]:
df_pca_light = X_top50light
df_pca_light = df_pca_light.join(y)

In [35]:
from sklearn.model_selection import train_test_split
X_pca_light = df_pca_light.drop('Label',axis=1)
Y_pca_light = df_pca_light['Label']
X_train, X_test, y_train, y_test = train_test_split(
    X_pca_light, Y_pca_light, test_size=0.2, random_state=42
)

In [36]:
from collections import Counter
print("Before oversampling:", Counter(y_train))

Before oversampling: Counter({0.0: 1277, 1.0: 643})


In [37]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_train_res, y_train_res = rus.fit_resample(X_train, y_train)
print("After undersampling:", Counter(y_train_res))

After undersampling: Counter({0.0: 643, 1.0: 643})


In [38]:
from imblearn.over_sampling import KMeansSMOTE, ADASYN
smote = KMeansSMOTE(sampling_strategy={1:1400},k_neighbors=5,random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("After SMOTE:", Counter(y_train_smote))

After SMOTE: Counter({1.0: 1402, 0.0: 1277})


In [39]:
from imblearn.over_sampling import ADASYN
adasyn = ADASYN(sampling_strategy={1: 1400}, n_neighbors=5, random_state=42)
X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)
print("After ADASYN:", Counter(y_train_adasyn))

After ADASYN: Counter({1.0: 1411, 0.0: 1277})


In [40]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

estimators1 = [
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=10, random_state=42)),
    ('svc', SVC(probability=True, random_state=42)),
    ('xgb',XGBClassifier(n_estimators=100,learning_rate=0.3))
]

final_estimator1 = GradientBoostingClassifier()
stacking_clf1 = StackingClassifier(
    estimators=estimators1,
    final_estimator=final_estimator1,
    cv=10
)
stacking_clf1.fit(X_train_smote, y_train_smote)
y_pred1 = stacking_clf1.predict(X_test)
print("Stacking Accuracy(With SMOTE):", accuracy_score(y_test, y_pred1))
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred1))

Stacking Accuracy(With SMOTE): 0.9854166666666667
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99       323
         1.0       0.97      0.98      0.98       157

    accuracy                           0.99       480
   macro avg       0.98      0.98      0.98       480
weighted avg       0.99      0.99      0.99       480



In [41]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

estimators2 = [
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=10, random_state=42)),
    ('xgb',XGBClassifier(n_estimators=100,learning_rate=0.3)),
    ('svc', SVC(probability=True, random_state=42)),
]

final_estimator2 = GradientBoostingClassifier()
stacking_clf2 = StackingClassifier(
    estimators=estimators2,
    final_estimator=final_estimator2,
    cv=10
)
stacking_clf2.fit(X_train_adasyn, y_train_adasyn)
y_pred2 = stacking_clf2.predict(X_test)
print("Stacking Accuracy(With ADASYN):", accuracy_score(y_test, y_pred2))
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred2))


Stacking Accuracy(With ADASYN): 0.99375
              precision    recall  f1-score   support

         0.0       1.00      0.99      1.00       323
         1.0       0.98      1.00      0.99       157

    accuracy                           0.99       480
   macro avg       0.99      1.00      0.99       480
weighted avg       0.99      0.99      0.99       480



In [42]:
estimators3 = [
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=10, random_state=42)),
    ('svc', SVC(probability=True, random_state=42)),
    ('xgb',XGBClassifier(n_estimators=100,learning_rate=0.3))
]

final_estimator3 = GradientBoostingClassifier()
stacking_clf3 = StackingClassifier(
    estimators=estimators3,
    final_estimator=final_estimator3,
    cv=10
)
stacking_clf3.fit(X_train_res, y_train_res)
y_pred3 = stacking_clf3.predict(X_test)
print("Stacking Accuracy(With Undersampling):", accuracy_score(y_test, y_pred3))
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred3))

Stacking Accuracy(With Undersampling): 0.98125
              precision    recall  f1-score   support

         0.0       0.99      0.98      0.99       323
         1.0       0.97      0.97      0.97       157

    accuracy                           0.98       480
   macro avg       0.98      0.98      0.98       480
weighted avg       0.98      0.98      0.98       480



In [43]:
from sklearn.model_selection import RandomizedSearchCV
estimators2 = [
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=10, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.3)),
    ('svc', SVC(probability=True, random_state=42)),
]
param_grids = {
    'rf': {
        'n_estimators': [10, 20],
        'max_depth': [None, 10]
    },
    'gb': {
        'n_estimators': [10, 20],
        'learning_rate': [0.1],
        'max_depth': [3]
    },
    'xgb': {
        'n_estimators': [50, 100],
        'learning_rate': [0.1, 0.3],
        'max_depth': [3]
    },
    'svc': {
        'C': [0.1, 1],
        'kernel': ['linear', 'rbf'],
        'gamma': ['scale']
    }
}
from sklearn.base import clone
tuned_estimators2 = []
for name, model in estimators2:
    print(f"Tuning {name}...")
    grid = RandomizedSearchCV(
        estimator=clone(model),
        param_distributions=param_grids[name],
        n_iter=2,      
        cv=2,         
        scoring='accuracy',
        n_jobs=-1,
        random_state=42
    )
    grid.fit(X_train_adasyn, y_train_adasyn)
    print(f"Best params for {name}: {grid.best_params_}")
    tuned_estimators2.append((name, grid.best_estimator_))

Tuning rf...
Best params for rf: {'n_estimators': 20, 'max_depth': 10}
Tuning gb...
Best params for gb: {'n_estimators': 20, 'max_depth': 3, 'learning_rate': 0.1}
Tuning xgb...
Best params for xgb: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.3}
Tuning svc...
Best params for svc: {'kernel': 'rbf', 'gamma': 'scale', 'C': 1}


In [44]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
stacking = StackingClassifier(estimators=tuned_estimators2, final_estimator=LogisticRegression())
stacking.fit(X_train_adasyn, y_train_adasyn)

StackingClassifier(estimators=[('rf',
                                RandomForestClassifier(max_depth=10,
                                                       n_estimators=20,
                                                       random_state=42)),
                               ('gb',
                                GradientBoostingClassifier(n_estimators=20,
                                                           random_state=42)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enabl...
                                              learning_rate=0.3, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=3,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=100, n_jobs=None,
                                              num_parallel_tree=None, ...)),
                               ('svc',
                                SVC(C=1, probability=True, random_state=42))],
                   final_estimator=LogisticRegression())

In [45]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
y_pred = stacking.predict(X_test)
y_proba = stacking.predict_proba(X_test)[:, 1] 
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_proba)
print("=== Classification Report ===")
print(classification_report(y_test, y_pred))
print("=== Evaluation Metrics ===")
print(f"Accuracy      : {acc:.4f}")
print(f"Precision     : {prec:.4f}")
print(f"Recall        : {rec:.4f}")
print(f"F1 Score      : {f1:.4f}")
print(f"ROC AUC Score : {roc:.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

         0.0       1.00      0.99      1.00       323
         1.0       0.98      1.00      0.99       157

    accuracy                           0.99       480
   macro avg       0.99      1.00      0.99       480
weighted avg       0.99      0.99      0.99       480

=== Evaluation Metrics ===
Accuracy      : 0.9938
Precision     : 0.9812
Recall        : 1.0000
F1 Score      : 0.9905
ROC AUC Score : 0.9998


In [48]:
from sklearn.model_selection import cross_val_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
estimators2 = [
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=10, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=100, learning_rate=0.3)),
    ('svc', SVC(probability=True, random_state=42)),
]
search_spaces = {
    'rf': {
        'n_estimators': hp.choice('n_estimators_rf', [10, 50, 100]),
        'max_depth': hp.choice('max_depth_rf', [None, 10, 20]),
        'max_features': hp.choice('max_features_rf', ['sqrt', 'log2'])
    },
    'gb': {
        'n_estimators': hp.choice('n_estimators_gb', [10, 50, 100]),
        'learning_rate': hp.uniform('learning_rate_gb', 0.05, 0.3),
        'max_depth': hp.choice('max_depth_gb', [3, 5, 7])
    },
    'xgb': {
        'n_estimators': hp.choice('n_estimators_xgb', [50, 100]),
        'learning_rate': hp.uniform('learning_rate_xgb', 0.05, 0.3),
        'max_depth': hp.choice('max_depth_xgb', [3, 5, 7])
    },
    'svc': {
        'C': hp.uniform('C_svc', 0.1, 10),
        'kernel': hp.choice('kernel_svc', ['linear', 'rbf']),
        'gamma': hp.choice('gamma_svc', ['scale', 'auto'])
    }
}
def build_model(name, params):
    if name == 'rf':
        return RandomForestClassifier(**params, random_state=42)
    elif name == 'gb':
        return GradientBoostingClassifier(**params, random_state=42)
    elif name == 'xgb':
        return XGBClassifier(**params, use_Label_encoder=False, eval_metric='logloss')
    elif name == 'svc':
        return SVC(**params, probability=True, random_state=42)
def make_objective(name):
    def objective(params):
        model = build_model(name, params)
        score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
        return {'loss': -score, 'status': STATUS_OK}
    return objective

tuned_estimators = []
max_evals = 5
for name, _ in estimators2:
    print(f"Tuning {name}...")
    trials = Trials()
    best_params = fmin(
        fn=make_objective(name),
        space=search_spaces[name],
        algo=tpe.suggest,
        max_evals=max_evals,
        trials=trials,
        rstate=np.random.default_rng(42)
    )
    def map_choice(key, choices):
        return choices[best_params[key]]

    if name == 'rf':
        final_params = {
            'n_estimators': [10, 50, 100][best_params['n_estimators_rf']],
            'max_depth': [None, 10, 20][best_params['max_depth_rf']],
            'max_features': ['sqrt', 'log2'][best_params['max_features_rf']]
        }
    elif name == 'gb':
        final_params = {
            'n_estimators': [10, 50, 100][best_params['n_estimators_gb']],
            'learning_rate': best_params['learning_rate_gb'],
            'max_depth': [3, 5, 7][best_params['max_depth_gb']]
        }
    elif name == 'xgb':
        final_params = {
            'n_estimators': [50, 100][best_params['n_estimators_xgb']],
            'learning_rate': best_params['learning_rate_xgb'],
            'max_depth': [3, 5, 7][best_params['max_depth_xgb']]
        }
    elif name == 'svc':
        final_params = {
            'C': best_params['C_svc'],
            'kernel': ['linear', 'rbf'][best_params['kernel_svc']],
            'gamma': ['scale', 'auto'][best_params['gamma_svc']]
        }
    model = build_model(name, final_params)
    tuned_estimators.append((name, model))
print("\nAll models tuned using Hyperopt!")

Tuning rf...
100%|█████████████████████████████████| 5/5 [00:01<00:00,  4.73trial/s, best loss: -0.9713541666666666]
Tuning gb...
100%|█████████████████████████████████| 5/5 [00:02<00:00,  2.31trial/s, best loss: -0.9776041666666666]
Tuning xgb...
 20%|██████▌                          | 1/5 [00:00<00:00,  8.95trial/s, best loss: -0.9791666666666666]

/home/amogh-asus/miniconda3/envs/cuda_env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:23:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_Label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

/home/amogh-asus/miniconda3/envs/cuda_env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:23:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_Label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

/home/amogh-asus/miniconda3/envs/cuda_env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:23:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_Label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

/home/amogh-asus/miniconda3/envs/cuda_env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:23:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_Label_encoder" } are not used.

  bst.update(dtra

 40%|█████████████▏                   | 2/5 [00:00<00:00,  7.49trial/s, best loss: -0.9812500000000001]

/home/amogh-asus/miniconda3/envs/cuda_env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:23:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_Label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

/home/amogh-asus/miniconda3/envs/cuda_env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:23:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_Label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

/home/amogh-asus/miniconda3/envs/cuda_env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:23:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_Label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

/home/amogh-asus/miniconda3/envs/cuda_env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:23:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_Label_encoder" } are not used.

  bst.update(dtra

100%|█████████████████████████████████| 5/5 [00:00<00:00,  8.19trial/s, best loss: -0.9812500000000001]
Tuning svc...
  0%|                                                            | 0/5 [00:00<?, ?trial/s, best loss=?]

/home/amogh-asus/miniconda3/envs/cuda_env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:23:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_Label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

/home/amogh-asus/miniconda3/envs/cuda_env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:23:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_Label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

/home/amogh-asus/miniconda3/envs/cuda_env/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:23:08] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_Label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



100%|█████████████████████████████████| 5/5 [00:00<00:00, 15.02trial/s, best loss: -0.9963541666666668]

All models tuned using Hyperopt!


In [32]:
import time
start = time.time()
stacking_clf2.fit(X_train_adasyn, y_train_adasyn)
end = time.time()
print(f"{end - start:.2f}")


2.34


In [49]:
model999 = LogisticRegression()
model999.fit(X_train_adasyn,y_train_adasyn)
y_pred999 = model999.predict(X_test)
print("Stacking Accuracy(With Adasyn):", accuracy_score(y_test, y_pred999))
print(classification_report(y_test, y_pred999))

Stacking Accuracy(With Adasyn): 0.9958333333333333
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       323
         1.0       0.99      0.99      0.99       157

    accuracy                           1.00       480
   macro avg       1.00      1.00      1.00       480
weighted avg       1.00      1.00      1.00       480

